In [0]:
from pyspark.sql import functions as F

df = spark.table("personal_projects.information_retail_schema.silver_online_retail_clean")

# Monthly revenue by country
monthly_revenue = (df
    .withColumn("YearMonth", F.date_format("InvoiceDate", "yyyy-MM"))
    .groupBy("YearMonth", "Country")
    .agg(F.round(F.sum("Revenue"), 2).alias("TotalRevenue"),
         F.countDistinct("Invoice").alias("NumOrders")))

monthly_revenue.write.format("delta").mode("overwrite").saveAsTable("personal_projects.information_retail_schema.gold_monthly_revenue_by_country")

# Customer lifetime value
customer_ltv = (df.groupBy("CustomerID")
    .agg(F.round(F.sum("Revenue"),2).alias("TotalSpend"),
         F.countDistinct("Invoice").alias("NumOrders"),
         F.max("InvoiceDate").alias("LastPurchase"))
    .orderBy(F.desc("TotalSpend")))

customer_ltv.write.format("delta").mode("overwrite").saveAsTable("personal_projects.information_retail_schema.gold_customer_ltv")

# Top products
top_products = (df.groupBy("StockCode", "Description")
    .agg(F.round(F.sum("Revenue"),2).alias("TotalRevenue"),
         F.sum("Quantity").alias("TotalQuantity"))
    .orderBy(F.desc("TotalRevenue")))

top_products.write.format("delta").mode("overwrite").saveAsTable("personal_projects.information_retail_schema.gold_top_products")